In [ ]:
# RNA velocity with dynamo.
import scanpy as sc
import numpy as np
import pandas as pd  
import matplotlib.pyplot as plt
import matplotlib
import anndata as ad
import scvelo as scv
import palantir as pl
import dynamo as dyn

In [ ]:
dyn.session_info()

In [ ]:
# Load data
adata = sc.read('/home/yanjun2/scanpy/8_figures/inte_palantir.h5ad')

In [ ]:
sc.settings.figdir = "/home/yanjun2/scanpy/8_figures"
sc.set_figure_params(scanpy=True, dpi=300, dpi_save=300, 
                         frameon=True, vector_friendly=True, fontsize=10, figsize= (5,5), color_map=None, format='pdf', 
                         facecolor=None, transparent=False, ipython_format='png2x')

In [ ]:
ldata1 = scv.read('/home/yc/Desktop/loom/R1_D1MDI_D1A.loom', cache=True)
ldata2 = scv.read('/home/yc/Desktop/loom/R1_D2MDI_D2A.loom', cache=True)
ldata3 = scv.read('/home/yc/Desktop/loom/R1_D3MDI_D3A.loom', cache=True)
ldata4 = scv.read('/home/yc/Desktop/loom/R1_D4MDI_D4A.loom', cache=True)
ldata5 = scv.read('/home/yc/Desktop/loom/R1_D5MDI_D5A.loom', cache=True)
#ldata6 = scv.read('/home/yc/Desktop/loom/R1_DMo_D0M.loom', cache=True)

In [ ]:
def clean_cell_ids(ldata):
    ldata.obs.index = ldata.obs.index.str.replace(r'^.*:', '', regex=True)
    ldata.obs.index = ldata.obs.index.str.replace(r'x$', '', regex=True)
    return ldata
ldata1 = clean_cell_ids(ldata1)
ldata2 = clean_cell_ids(ldata2)
ldata3 = clean_cell_ids(ldata3)
ldata4 = clean_cell_ids(ldata4)
ldata5 = clean_cell_ids(ldata5)

In [ ]:
ldata1.var_names_make_unique()
ldata2.var_names_make_unique()
ldata3.var_names_make_unique()
ldata4.var_names_make_unique()
ldata5.var_names_make_unique()

In [ ]:
ldata = ldata1.concatenate([ldata2, ldata3, ldata4, ldata5
                            ])

In [ ]:
adata.obs_names = adata.obs_names.str.split('-').str[0]

In [ ]:
adata = scv.utils.merge(adata, ldata)

In [ ]:
adata.X = adata.layers['spliced']
adata.X = adata.layers['unspliced']

In [ ]:
dyn.tl.dynamics(adata)
dyn.tl.moments(adata)

In [ ]:
dyn.tl.gene_wise_confidence(adata, group='leiden_res_0_8', lineage_dict={'Mac1': ['Mac7']})

In [ ]:
dyn.tl.reduceDimension(adata)

In [ ]:
dyn.tl.reduceDimension(adata, basis='umap', n_pca_components=30)

In [ ]:
dyn.tl.cell_velocities(adata, basis="pca")

In [ ]:
dyn.tl.cell_velocities(adata, basis='umap')

In [ ]:
dyn.pl.streamline_plot(adata, color=["leiden_res_0_8"], basis="umap", show_legend="on data", show_arrowed_spines=True)

In [ ]:
dyn.tl.cell_wise_confidence(adata)

In [ ]:
dyn.pl.cell_wise_vectors(
    adata,
    color=["leiden_res_0_8"],
    basis="umap",
    show_legend="on data",
    quiver_length=6,
    quiver_size=6,
    figsize=(8, 6),
    show_arrowed_spines=False,
)

In [ ]:
progenitor = adata.obs_names[adata.obs.leiden_res_0_8.isin(["Mac1"])]
len(progenitor)

In [ ]:
dyn.vf.VectorField(adata, basis="pca", pot_curl_div=True)
dyn.vf.VectorField(adata, basis="umap", pot_curl_div=True)

dyn.vf.speed(adata, basis="pca")
dyn.vf.divergence(adata, basis="pca")
dyn.vf.acceleration(adata, basis="pca")
dyn.vf.curl(adata, basis="umap")


In [ ]:
dyn.configuration.set_figure_params("dynamo", background="black")

import matplotlib.pyplot as plt

fig1, fig1_axes = plt.subplots(ncols=2, nrows=2, constrained_layout=True, figsize=(12, 8))
dyn.pl.cell_wise_vectors(
    adata,
    color="pca_ddhodge_potential",
    pointsize=0.1,
    alpha=0.7,
    ax=fig1_axes[0, 0],
    quiver_length=6,
    quiver_size=6,
    save_show_or_return="return",
    background="black",
)
dyn.pl.grid_vectors(
    adata,
    color="speed_pca",
    basis="umap",
    ax=fig1_axes[0, 1],
    quiver_length=12,
    quiver_size=12,
    save_show_or_return="return",
    background="black",
)
dyn.pl.streamline_plot(
    adata, color="divergence_pca", basis="umap", ax=fig1_axes[1, 0], save_show_or_return="return", background="black"
)
dyn.pl.streamline_plot(
    adata, color="acceleration_pca", basis="umap", ax=fig1_axes[1, 1], save_show_or_return="return", background="black"
)
plt.show()

In [ ]:
dyn.pl.topography(
    adata, basis="umap", background="white", color=["leiden_res_0_8"], streamline_color="black", show_legend="on data"
)


In [ ]:
dyn.configuration.set_figure_params("dynamo", background="white")
dyn.pd.state_graph(adata, group='leiden_res_0_8', basis='pca', method='vf')
dyn.pl.state_graph(adata,
                   color=['leiden_res_0_8'],
                   group='leiden_res_0_8',
                   basis='umap',
                   show_legend='on data',
                   method='vf');



In [ ]:
markers = [
    'S100A8',
    'C1QA',
    'LYN',
    'LYVE1'
]

In [ ]:
markers

In [ ]:
dyn.vf.jacobian(adata, regulators= markers)

In [ ]:
# Neighbours, UMAP and clustering
scv.pp.filter_and_normalize(adata)
sc.pp.neighbors(adata, n_pcs=30, n_neighbors=30, random_state=0)
scv.pp.moments(adata, n_pcs=None, n_neighbors=None)

In [ ]:
scv.tl.velocity(adata, mode = 'stochastic')
scv.tl.velocity_graph(adata)

In [ ]:
adata.obsm['exp_diffmap'] = adata.obsm['X_diffmap']

In [ ]:
scv.pl.velocity_embedding_stream(adata, basis='umap', color = 'leiden_res_0_8', legend_loc='right margin', arrow_size = 0.5, 
                                 size = 100#, save='velo_mdixwithouit.pdf'
                                )

In [ ]:
scv.tl.velocity_pseudotime(adata)
scv.pl.scatter(adata, color='velocity_pseudotime', cmap='RdBu_r')

In [ ]:
sc.pl.umap(adata, color='velocity_pseudotime', color_map='RdBu_r', size = 50)

In [ ]:
sc.pl.umap(adata, color='palantir_pseudotime', color_map='RdBu_r', size = 50)

In [ ]:
sc.pl.scatter(adata, x = 'palantir_pseudotime', y = 'velocity_pseudotime', color = 'leiden_res_0_8', size = 20)

In [ ]:
sc.tl.embedding_density(adata, basis='umap', groupby='condition')
sc.pl.embedding_density(adata, basis='umap', key='umap_density_condition')

In [ ]:
scv.pl.velocity_embedding_stream(adata, basis='diffmap', color = 'leiden_res_0_8', legend_loc='right margin', arrow_size = 0.5, 
                                 size = 100#, save='velo_mdixwithouit.pdf'
                                )

In [ ]:
scv.pl.velocity_embedding_stream(adata, basis='diffmap', color = 'velocity_pseudotime', legend_loc='right margin', arrow_size = 0.5, 
                                 size = 100, cmap = 'RdBu_r'
                                )

In [ ]:
scv.pl.velocity_embedding_stream(adata, basis='diffmap', color = 'palantir_pseudotime', legend_loc='right margin', arrow_size = 0.5, 
                                 size = 100, cmap = 'RdBu_r'
                                )